In [1]:
import random
import copy
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.set_num_threads(
    max(1, torch.get_num_threads() - 1)
)

DATA_FILE = "../data/idm_pt.tsv"
MODELS_DIR = "../models/nb4/"

BSM_FEATURES = [
    "mH",
    "mA",
    "mHp",
    "l2",
    "l345",
]

TARGET_COLUMNS = [
    "Tcrit",
    "Tnuc",
    "log_alpha",
    "log_beta/H",
]

N_FEATURES = len(BSM_FEATURES)
N_TARGETS = len(TARGET_COLUMNS)

In [3]:
df = pd.read_csv(
    DATA_FILE,
    sep="\t",
)

df_clean = df.copy()

df_clean["Tcrit"] = df_clean["Tcrit"].mask(
    df_clean["Tcrit"] == -999,
    df_clean["Tnuc"],
)

df_clean = df_clean[
    df_clean["beta/H"] != -999
].copy()

df_clean["log_alpha"] = np.log10(
    df_clean["alpha"]
)

df_clean["log_beta/H"] = np.log10(
    df_clean["beta/H"]
)

df_ml = df_clean[
    df_clean["transition_pattern"] == "oh1"
].copy()

required_columns = (
    BSM_FEATURES + TARGET_COLUMNS
)

finite_mask = np.isfinite(
    df_ml[required_columns]
    .to_numpy(dtype=np.float64)
).all(axis=1)

df_ml = df_ml.loc[finite_mask].copy()

print("Final training sample:", df_ml.shape)

Final training sample: (50297, 20)


In [4]:
X = df_ml[
    BSM_FEATURES
].to_numpy(dtype=np.float32)

y = df_ml[
    TARGET_COLUMNS
].to_numpy(dtype=np.float32)

In [5]:
feature_scaler = StandardScaler()

X_scaled = feature_scaler.fit_transform(X)

target_scaler = StandardScaler()

y_scaled = target_scaler.fit_transform(y)

In [6]:
class PTRegressor(nn.Module):

    def __init__(self, n_inputs, n_outputs):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_inputs, 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, n_outputs),
        )

    def forward(self, x):
        return self.network(x)

In [7]:
X_tensor = torch.tensor(
    X_scaled,
    dtype=torch.float32,
)

y_tensor = torch.tensor(
    y_scaled,
    dtype=torch.float32,
)

dataset = TensorDataset(
    X_tensor,
    y_tensor,
)

loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,
)

In [8]:
final_model = PTRegressor(
    n_inputs=N_FEATURES,
    n_outputs=N_TARGETS,
).to(DEVICE)

loss_function = nn.MSELoss()

optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

MAX_EPOCHS = 300

training_losses = []

for epoch in range(1, MAX_EPOCHS + 1):

    final_model.train()

    batch_losses = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        prediction = final_model(X_batch)

        loss = loss_function(
            prediction,
            y_batch,
        )

        loss.backward()
        optimizer.step()

        batch_losses.append(
            loss.item()
        )

    mean_loss = np.mean(batch_losses)

    training_losses.append(mean_loss)

    if epoch == 1 or epoch % 10 == 0:

        print(
            f"Epoch {epoch:3d} | "
            f"training MSE = {mean_loss:.6f}"
        )

Epoch   1 | training MSE = 0.359226
Epoch  10 | training MSE = 0.004370
Epoch  20 | training MSE = 0.002733
Epoch  30 | training MSE = 0.002315
Epoch  40 | training MSE = 0.002192
Epoch  50 | training MSE = 0.001997
Epoch  60 | training MSE = 0.001888
Epoch  70 | training MSE = 0.001838
Epoch  80 | training MSE = 0.001806
Epoch  90 | training MSE = 0.001755
Epoch 100 | training MSE = 0.001719
Epoch 110 | training MSE = 0.001686
Epoch 120 | training MSE = 0.001661
Epoch 130 | training MSE = 0.001675
Epoch 140 | training MSE = 0.001641
Epoch 150 | training MSE = 0.001608
Epoch 160 | training MSE = 0.001672
Epoch 170 | training MSE = 0.001576
Epoch 180 | training MSE = 0.001594
Epoch 190 | training MSE = 0.001581
Epoch 200 | training MSE = 0.001567
Epoch 210 | training MSE = 0.001601
Epoch 220 | training MSE = 0.001562
Epoch 230 | training MSE = 0.001529
Epoch 240 | training MSE = 0.001542
Epoch 250 | training MSE = 0.001553
Epoch 260 | training MSE = 0.001515
Epoch 270 | training MSE = 0

In [11]:
torch.save(
    final_model.state_dict(),
    MODELS_DIR + "final_mlp_regressor.pt",
)

joblib.dump(
    feature_scaler,
    MODELS_DIR + "final_feature_scaler.pkl",
)

joblib.dump(
    target_scaler,
    MODELS_DIR + "final_target_scaler.pkl",
)

['../models/nb4/final_target_scaler.pkl']

In [ ]:
metadata = {
    "model": "PyTorch MLP",
    "input_features": BSM_FEATURES,
    "target_columns": TARGET_COLUMNS,
    "physical_targets": [
        "Tcrit",
        "Tnuc",
        "alpha",
        "beta/H",
    ],
    "log_transform": {
        "alpha": "log10(alpha)",
        "beta/H": "log10(beta/H)",
    },
    "architecture": [
        "Linear(5, 64)",
        "ReLU",
        "Linear(64, 64)",
        "ReLU",
        "Linear(64, 32)",
        "ReLU",
        "Linear(32, 4)",
    ],
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "batch_size": 512,
    "epochs": MAX_EPOCHS,
    "seed": SEED,
    "training_sample": "one-step FOPT transition_pattern == 'oh1'",
    "training_events": len(df_ml),
}